# Trajectory Optimization in Uncertain Environments
**Author:** Aditya Joshi | Student ID: 2530019

**Sections:**
1. Imports & Setup
2. Environment Preview
3. Nominal iLQR Trajectory
4. Controller Rollouts (with limited sensing)
5. Navigation Animation — dropdown to switch controller
6. Tracking Error & Control Effort over Time
7. Metrics Comparison

---
## 1. Imports & Setup

In [11]:
import sys, os, functools
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown
%matplotlib widget

# ── Environment
from environment.dynamics     import ContinuousTimeSpaceshipDynamics
from environment.integrators  import RK4Integrator
from environment.asteroid_env import Environment
from environment.ilqr         import (
    iterative_linear_quadratic_regulator,
    TotalCost, RunningCost, FullHorizonTerminalCost,
    compute_nominal_trajectory,
)

# ── Controllers
from controllers.pid            import PositionPIDController, FullStatePIDController
from controllers.state_feedback import StateFeedbackController, InfiniteHorizonLQRController

# ── Sensing simulation
from utils.sensing  import simulate_with_sensing
from utils.metrics  import evaluate_controller

# ── Config
from config import (
    T, dt, start_state, goal_position,
    Q, R, STATE_DIM, CONTROL_DIM,
    RANDOM_SEED, NUM_ASTEROIDS, LIMITED_SENSING,
    R_LIMIT, A_LIMIT,
)

np.random.seed(RANDOM_SEED)
dynamics = RK4Integrator(ContinuousTimeSpaceshipDynamics(), dt)
env      = Environment.create(NUM_ASTEROIDS)

COLORS = {
    'PID (position)':   '#e74c3c',
    'PID (full-state)': '#e67e22',
    'State Feedback':   '#2ecc71',
    'Inf-Horizon LQR':  '#3498db',
}

print('All imports OK')

ImportError: cannot import name 'RunningCost' from 'environment.ilqr' (c:\Users\Adity\Documents\trajectory_optmization_ME548\trajectory_optimization\environment\ilqr.py)

---
## 2. Environment Preview
Scrub the slider to watch asteroids move over time.

In [ ]:
fig_env, ax_env = env.plot(state=start_state)
ax_env.scatter(*goal_position, s=120, marker='*', color='gold', zorder=20, label='Goal')
ax_env.legend(loc='upper left')

@interact(k=IntSlider(min=0, max=T, step=1, value=0, description='Step k'))
def animate_env(k=0):
    env.at_time(k * dt).plot(state=start_state, ax=ax_env)
    ax_env.set_title(f'Asteroid Environment  |  t = {k * dt:.1f}s')
    fig_env.canvas.draw_idle()

---
## 3. Nominal iLQR Trajectory
Two-stage warm-start: solve on empty env first, then full asteroid field.
This reference trajectory is tracked by all controllers.

In [ ]:
solution = compute_nominal_trajectory(
    dynamics, env, start_state, goal_position, T, verbose=True
)
xs_nom, us_nom = solution['optimal_trajectory']

In [ ]:
# ── Animate nominal trajectory
fig_nom, ax_nom = env.plot(state=start_state, plan=xs_nom, history=xs_nom[:1])
ax_nom.scatter(*goal_position, s=120, marker='*', color='gold', zorder=20, label='Goal')
ax_nom.legend(loc='upper left')

@interact(k=IntSlider(min=0, max=T, step=1, value=0, description='Step k'))
def animate_nominal(k=0):
    thrust = float(us_nom[min(k, T-1), 1]) / 4.0
    env.at_time(k * dt).plot(
        state         = xs_nom[k],
        plan          = xs_nom[k:],
        history       = xs_nom[:k+1],
        scaled_thrust = thrust,
        sensor        = False,
        ax            = ax_nom,
    )
    ax_nom.set_title(f'Nominal iLQR  |  t={k*dt:.1f}s  |  thrust={us_nom[min(k,T-1),1]:.2f}')
    fig_nom.canvas.draw_idle()

---
## 4. Controller Rollouts with Limited Sensing
Each controller navigates with `sensing_radius` visibility only.
Asteroids outside that radius are invisible — the ship must avoid
what it can see and track the nominal path otherwise.

In [ ]:
# ── Instantiate all controllers
pid_pos = PositionPIDController(
    Kp_pos=1.0, Ki_pos=0.01, Kd_pos=0.5,
    Kp_heading=2.0, Kp_speed=1.0, Kd_speed=0.3,
    r_limit=R_LIMIT, a_limit=A_LIMIT, dt=dt,
)

pid_full = FullStatePIDController(
    Kp_x=1.0, Kd_x=0.5, Ki_x=0.01,
    Kp_y=1.0, Kd_y=0.5, Ki_y=0.01,
    Kp_th=2.0, Kd_th=0.3,
    Kp_vx=0.5, Kd_vx=0.1,
    Kp_vy=0.5, Kd_vy=0.1,
    r_limit=R_LIMIT, a_limit=A_LIMIT, dt=dt,
)

sf_ctrl = StateFeedbackController(dynamics=dynamics, Q=Q, R=R,
                                   r_limit=R_LIMIT, a_limit=A_LIMIT)
sf_ctrl.prepare(xs_nom, us_nom)

lqr_ctrl = InfiniteHorizonLQRController(dynamics=dynamics, Q=Q, R=R,
                                         r_limit=R_LIMIT, a_limit=A_LIMIT)
lqr_ctrl.prepare(xs_nom, us_nom)

controllers = {
    'PID (position)':   pid_pos,
    'PID (full-state)': pid_full,
    'State Feedback':   sf_ctrl,
    'Inf-Horizon LQR':  lqr_ctrl,
}
print('Controllers ready.')

In [ ]:
# ── Run all controllers with limited sensing
# simulate_with_sensing applies env.sense() at each step so the
# controller only reacts to asteroids within sensing_radius.
results = {}
for name, ctrl in controllers.items():
    print(f'Running: {name} ...', end=' ', flush=True)
    res = simulate_with_sensing(
        ctrl, start_state, env, dynamics,
        xs_nom, us_nom, goal_position,
        T=T, dt=dt, limited_sensing=LIMITED_SENSING,
        verbose=False,
    )
    results[name] = res
    status = '✓' if res['success'] else ('collision' if res['collision'] else 'timeout')
    print(f'{status}  |  track={np.mean(res["tracking_errors"]):.2f}m  '
          f'effort={np.mean(res["control_efforts"]):.2f}  '
          f'time={res["traversal_time"]} steps')

---
## 5. Navigation Animation
Use the **dropdown** to switch controller.
Use the **slider** to scrub through time.

The dashed circle is the sensing radius — only asteroids inside it are visible to the controller.
Green line = nominal reference. Colored trail = actual path taken.

In [ ]:
# ── Setup the figure once — reuse across slider/dropdown updates
_first = list(results.keys())[0]
fig_anim, ax_anim = env.plot(
    state   = start_state,
    plan    = xs_nom,
    history = results[_first]['history'][:1],
)
ax_anim.scatter(*goal_position, s=150, marker='*', color='gold', zorder=25, label='Goal')
# Actual path line — updated each frame
_path_line, = ax_anim.plot([], [], lw=2, zorder=20, color=COLORS[_first], label=_first)
ax_anim.legend(loc='upper left', fontsize=8)

@interact(
    controller=Dropdown(
        options=list(results.keys()),
        value=_first,
        description='Controller',
        style={'description_width': 'initial'},
    ),
    k=IntSlider(min=0, max=T, step=1, value=0, description='Step k'),
)
def animate_navigation(controller=_first, k=0):
    res      = results[controller]
    history  = res['history']       # (T+1, 5)
    controls = res['controls']      # (T,   2)

    thrust = float(np.abs(controls[min(k, T-1), 1])) / A_LIMIT

    # Update environment plot: spacecraft pose, sensing circle, asteroid positions
    env.at_time(k * dt).plot(
        state         = history[k],
        plan          = xs_nom,           # green nominal ahead
        history       = history[:k+1],    # blue trail
        scaled_thrust = thrust,
        sensor        = LIMITED_SENSING,  # show sensing circle
        ax            = ax_anim,
    )

    # Overlay actual path in controller colour
    _path_line.set_data(history[:k+1, 0], history[:k+1, 1])
    _path_line.set_color(COLORS[controller])
    _path_line.set_label(controller)

    # Status
    track_err = float(np.linalg.norm(history[k, :2] - xs_nom[k, :2]))
    visible   = res['sensed_asteroids'][min(k, T-1)]
    status    = '✓ goal' if res['success'] else ('✗ collision' if res['collision'] else '...')

    ax_anim.set_title(
        f"{controller}  |  t={k*dt:.1f}s  |  "
        f"track err={track_err:.2f}m  |  "
        f"visible asteroids={visible}  |  "
        f"thrust={controls[min(k,T-1),1]:.2f}  |  {status}"
    )
    ax_anim.legend(loc='upper left', fontsize=8)
    fig_anim.canvas.draw_idle()

---
## 6. Tracking Error & Control Effort over Time

In [ ]:
steps = np.arange(T) * dt
fig_ts, (ax_track, ax_effort, ax_sens) = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

for name, res in results.items():
    ax_track.plot(steps, res['tracking_errors'],
                  color=COLORS[name], label=name, lw=1.5)
    ax_effort.plot(steps, res['control_efforts'],
                   color=COLORS[name], label=name, lw=1.5)
    ax_sens.plot(steps, res['sensed_asteroids'],
                 color=COLORS[name], label=name, lw=1.5)

ax_track.set_ylabel('Tracking Error (m)')
ax_track.set_title('Position Tracking Error over Time')
ax_track.legend(fontsize=8); ax_track.grid(alpha=0.3)

ax_effort.set_ylabel('Control Effort (||u||²)')
ax_effort.set_title('Control Effort over Time  (fuel proxy)')
ax_effort.legend(fontsize=8); ax_effort.grid(alpha=0.3)

ax_sens.set_ylabel('Visible Asteroids')
ax_sens.set_xlabel('Time (s)')
ax_sens.set_title('Asteroids within Sensing Radius over Time')
ax_sens.legend(fontsize=8); ax_sens.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Metrics Comparison
20-trial benchmark across random asteroid configurations.

In [ ]:
from utils.metrics import evaluate_controller

NUM_TRIALS = 20
benchmark_results = {}

for name, ctrl in controllers.items():
    print(f'\nEvaluating: {name}')
    if hasattr(ctrl, 'prepare'):
        ctrl.prepare(xs_nom, us_nom)
    metrics = evaluate_controller(
        ctrl, dynamics, xs_nom, us_nom,
        start_state, goal_position,
        T, dt,
        num_asteroids = NUM_ASTEROIDS,
        num_trials    = NUM_TRIALS,
        density_label = name,
        verbose       = True,
    )
    benchmark_results[name] = metrics

print('\nDone.')

In [ ]:
names  = list(benchmark_results.keys())
colors = [COLORS[n] for n in names]
x      = np.arange(len(names))
w      = 0.6

fig_bar, axes = plt.subplots(1, 4, figsize=(16, 5))

# Success rate
sr = [benchmark_results[n]['success_rate'] * 100 for n in names]
axes[0].bar(x, sr, color=colors, width=w, edgecolor='white')
axes[0].set_title('Success Rate (%)')
axes[0].set_ylim(0, 110)
for i, v in enumerate(sr):
    axes[0].text(i, v+1.5, f'{v:.0f}%', ha='center', fontsize=8)

# Tracking error
axes[1].bar(x,
    [benchmark_results[n]['tracking_error_mean'] for n in names],
    yerr=[benchmark_results[n]['tracking_error_std'] for n in names],
    color=colors, width=w, edgecolor='white', capsize=4)
axes[1].set_title('Tracking Error (m)')

# Control effort
axes[2].bar(x,
    [benchmark_results[n]['control_effort_mean'] for n in names],
    yerr=[benchmark_results[n]['control_effort_std'] for n in names],
    color=colors, width=w, edgecolor='white', capsize=4)
axes[2].set_title('Control Effort (fuel proxy)')

# Traversal time
axes[3].bar(x,
    [benchmark_results[n]['traversal_time_mean'] for n in names],
    yerr=[benchmark_results[n]['traversal_time_std'] for n in names],
    color=colors, width=w, edgecolor='white', capsize=4)
axes[3].axhline(T, color='gray', linestyle='--', lw=1, label=f'Max ({T})')
axes[3].set_title('Traversal Time (steps)')
axes[3].legend(fontsize=8)

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

fig_bar.suptitle(f'Controller Comparison  ({NUM_TRIALS} trials, {NUM_ASTEROIDS} asteroids)', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/controller_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/controller_comparison.png')

In [ ]:
# ── Summary table
print(f'\n{"Controller":<22} {"Success":>8} {"Track Err":>14} {"Ctrl Effort":>15} {"Time":>10}')
print('-' * 73)
for name in names:
    m = benchmark_results[name]
    print(
        f'{name:<22} '
        f'{m["success_rate"]*100:>7.0f}%  '
        f'{m["tracking_error_mean"]:>7.2f}±{m["tracking_error_std"]:.2f}  '
        f'{m["control_effort_mean"]:>9.2f}±{m["control_effort_std"]:.2f}  '
        f'{m["traversal_time_mean"]:>7.1f}±{m["traversal_time_std"]:.1f}'
    )